<a href="https://colab.research.google.com/github/fvarellalopes/clawsouls/blob/main/Z_Image_Turbo_4bit_jupyter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U git+https://github.com/huggingface/diffusers git+https://github.com/Disty0/sdnq

In [ ]:
import torch
import diffusers
from sdnq import SDNQConfig # import sdnq to register it into diffusers and transformers
from sdnq.loader import apply_sdnq_options_to_model

pipe = diffusers.ZImagePipeline.from_pretrained("Disty0/Z-Image-Turbo-SDNQ-uint4-svd-r32", torch_dtype=torch.float32, device_map="cuda")
pipe.transformer = apply_sdnq_options_to_model(pipe.transformer, use_quantized_matmul=True)
pipe.text_encoder = apply_sdnq_options_to_model(pipe.text_encoder, use_quantized_matmul=True)

In [ ]:
# prompt = "a grizzled 60-year-old mage, his face a striking fusion of ancient mysticism and cutting-edge cyberware, staring directly into the camera with piercing, bioluminescent eyes that flicker between arcane violet and cold machine blue. His silver-streaked beard is woven with delicate gold circuitry, pulsing faintly with energy, while the left side of his face transitions seamlessly into sleek, blackened metal plating, etched with glowing runes that hum with latent power. His robe, a tattered mix of enchanted fabric and nano-weave armor, clings to his broad shoulders, its frayed edges crackling with unstable magic. Behind him, a sprawling microchip cityscape throbs with neon-yellow circuit patterns against an abyssal black void, the labyrinthine pathways mirroring the intricate scars and implants across his weathered skin. The air around him shimmers with distortion—part holographic spell matrix, part overheating processor—as if reality itself struggles to contain him."
# image = pipe(
#     prompt=prompt,
#     height=1024,
#     width=1024,
#     num_inference_steps=9,
#     guidance_scale=0.0,
#     generator=torch.manual_seed(42),
# ).images[0]
# display(image)

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

In [ ]:
!pip install uvicorn


In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from io import BytesIO
import base64
import torch
import uvicorn
import threading
import time

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"])

TOKEN = 'cs-secret-2026'

class Req(BaseModel):
    prompt: str
    token: str

@app.get("/health")
async def health():
    return {"status": "ok"}

@app.post("/generate")
async def generate(req: Req):
    # Validação do token agora via parâmetro no corpo da request
    if req.token != TOKEN:
        raise HTTPException(status_code=401, detail="Invalid Token")

    image = pipe(prompt=req.prompt, height=1024, width=1024,
                 num_inference_steps=9, guidance_scale=0.0,
                 generator=torch.manual_seed(42)).images[0]
    buf = BytesIO()
    image.save(buf, format="PNG")
    return {"image": base64.b64encode(buf.getvalue()).decode()}

# Mata qualquer processo que esteja usando a porta 8081
!fuser -k 8081/tcp

def run_server():
    config = uvicorn.Config(app, host="0.0.0.0", port=8081, log_level="info")
    server = uvicorn.Server(config)
    server.run()

# Inicia em Thread para compartilhar memória
thread = threading.Thread(target=run_server, daemon=True)
thread.start()

time.sleep(2)
print("🚀 Servidor reiniciado. Token agora deve ser enviado no corpo do JSON.")

In [ ]:
import subprocess, re, time

CLOUDFLARE_TOKEN = "eyJhIjoiNmIxNmYwMzUwNjM5NWFhNjBjZjk2NzY0MDA2Y2I0MGUiLCJ0IjoiYzA1MzQ2NGEtNzBhYy00MmUwLWFkMjQtZDdiMDFkYzBmOGZhIiwicyI6Ik1qbGtOekEyWVRjdFkyWXdOQzAwTXpGa0xXSTNZV1V0WkdJek5HVTJNRGhrT0RVeCJ9"

cloudflared_proc = subprocess.Popen(
    ['cloudflared', 'tunnel',  '--url', 'http://localhost:8081'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

tunnel_url = None
print('⏳ Esperando URL do tunnel...')
for i in range(30):
    line = cloudflared_proc.stdout.readline().decode('utf-8', errors='replace')
    if not line:
        time.sleep(0.5)
        continue
    print(line.strip())
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    print(f"\n🌐 TÚNEL: {tunnel_url}")
else:
    print("❌ Não encontrou URL")


In [ ]:
import os

print("--- Conteúdo do uvicorn.log ---")
if os.path.exists("uvicorn.log"):
    with open("uvicorn.log", "r") as f:
        print(f.read())
else:
    print("Arquivo uvicorn.log não encontrado.")

print("\n--- Verificando processos ativos ---")
!ps aux | grep uvicorn | grep -v grep

print("\n--- Verificando portas ---")
!ss -tlnp | grep 8081